# MLP Regressor

Scikit-learn MLP regression with the shared selected features.

All forecasting notebooks use the same selected columns from `forecast_features.py`.

In [4]:
from pathlib import Path
import sys
from types import SimpleNamespace

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'backend' / 'ml').exists():
    repo_root = repo_root.parent

ml_dir = repo_root / 'backend' / 'ml'
if str(ml_dir) not in sys.path:
    sys.path.insert(0, str(ml_dir))

from forecast_features import (
    FEATURE_COLUMNS, NUMERIC_FEATURES, CATEGORICAL_FEATURES, REMOVED_FEATURES,
    FEATURE_IMPORTANCE_SOURCE, FEATURE_SELECTION_NOTES, TARGET_COLUMN
)

args = SimpleNamespace(
    input=str(repo_root / 'etl' / 'exports' / 'flights_features_all.csv'),
    output=str(repo_root / 'backend' / 'ml' / 'models' / 'mlp'),
    sample_size=0,
    limit_rows=None,
    test_size=0.2,
    random_state=42,
    n_jobs=1,
)

print('Target:', TARGET_COLUMN)
print('Selected features:', FEATURE_COLUMNS)


Target: price
Selected features: ['days_to_departure', 'stops', 'duration_minutes', 'distance_km', 'depart_hour', 'recent_price_trend_per_day', 'travel_class', 'airline', 'origin', 'destination', 'arrival_time', 'search_month', 'depart_dow', 'depart_month']


In [5]:
import pandas as pd

display(pd.DataFrame([{'feature': k, 'source_importance': v} for k, v in FEATURE_IMPORTANCE_SOURCE.items()]))
display(pd.DataFrame({'numeric_feature': NUMERIC_FEATURES}))
display(pd.DataFrame({'categorical_feature': CATEGORICAL_FEATURES}))
display(pd.DataFrame({'removed_feature': REMOVED_FEATURES}))
display(pd.DataFrame({'selection_note': FEATURE_SELECTION_NOTES}))


,feature,source_importance
0,travel_class,0.519
1,stops,0.153
2,airline,0.150
3,days_to_departure,0.038
4,origin,0.032
5,duration_minutes,0.028
6,distance_km,0.020
7,depart_hour,0.017
8,destination,0.014
9,recent_price_trend_per_day,0.010


,numeric_feature
0,days_to_departure
1,stops
2,duration_minutes
3,distance_km
4,depart_hour
5,recent_price_trend_per_day


,categorical_feature
0,travel_class
1,airline
2,origin
3,destination
4,arrival_time
5,search_month
6,depart_dow
7,depart_month


,removed_feature
0,flight_id
1,departure_date
2,departure_time
3,passengers_total
4,trip_type
5,origin_type
6,destination_type
7,depart_is_weekend
8,depart_season
9,search_dow


,selection_note
0,Keep depart_hour instead of departure_time bec...
1,Keep depart_dow instead of depart_is_weekend b...
2,Keep search_month instead of search_season bec...
3,Keep arrival_time because it describes arrival...
4,Remove price-derived columns to avoid target l...
5,Add recent_price_trend_per_day as a causal loc...


In [ ]:
from sklearn.neural_network import MLPRegressor
from forecast_model_utils import train_sklearn_forecaster

result = train_sklearn_forecaster(
    model_name='mlp',
    estimator=MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=300, early_stopping=True, random_state=args.random_state),
    input_path=args.input,
    output_dir=args.output,
    sample_size=args.sample_size,
    limit_rows=args.limit_rows,
    test_size=args.test_size,
    random_state=args.random_state,
)
print(result['metrics'])
display(result['feature_importance'].head(20))
display(result['predictions'].head(20))


c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:697: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
